# B3 – Gemini, ChatGPT, Claude, DeepSeek and the local bot as DDX engines

Same vignettes, same system prompt, same protocol, temperature 0. Results append to `results/runs.jsonl` (resumable). **Core = the 5 vignettes the practice requires; extra = more vignettes so B4 has statistical power.**

In [1]:
import sys, os, json, time, gc
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
import pandas as pd
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 200)
from vignettes import load_vignettes, sample_vignettes
from protocol import run_case, SYSTEM_DOCTOR
import clients, store, metrics


In [2]:
from pathlib import Path

# Move up from `notebook/` to project root once, even if re-run
if Path.cwd().name == "notebooks":
  os.chdir("..")

VIGNETTES = load_vignettes("data/Clinical_Vignettes_26_27.txt")
print(len(VIGNETTES), "clean 4-line vignettes")

117 clean 4-line vignettes


In [6]:
MODE = "single"          # "single" (recommended, comparable) or "multi" (3 turns)
N_CORE, N_EXTRA, SEED = 5, 25, 2026

# Model slugs are EXAMPLES: check them at https://openrouter.ai/models (the next cell tells you which are invalid)
OPENROUTER_BOTS = {
    "gemini":   "google/gemini-2.5-flash-lite",
    "chatgpt":  "openai/gpt-4o-mini",
    "claude":   "anthropic/claude-haiku-5.5",
    "deepseek": "deepseek/deepseek-chat",
}
LOCAL_BOT = ("local-qwen2.5-1.5b", "Qwen/Qwen2.5-1.5B-Instruct")   # the B1 chatbot as 5th bot
LOCAL_ON_EXTRA = False   # CPU is slow: local bot only on the 5 core vignettes unless True

print("invalid slugs:", clients.check_openrouter_models(OPENROUTER_BOTS.values()))
SAMPLE = sample_vignettes(VIGNETTES, N_CORE, N_EXTRA, SEED)
json.dump({"seed": SEED, "ids": [(v["id"], v["set"]) for v in SAMPLE], "mode": MODE,
           "openrouter_bots": OPENROUTER_BOTS, "local": LOCAL_BOT},
          open("results/run_config.json", "w"), indent=1)

invalid slugs: []


In [7]:
from tqdm.auto import tqdm
def run_bot(name, fn, vignettes):
    done = store.done_keys()
    for v in tqdm(vignettes, desc=name):
        if (name, v["id"], MODE) in done: continue
        try:
            store.save_run(name, run_case(fn, v, mode=MODE))
        except Exception as e:
            print("ERR", name, v["id"], repr(e)[:200])

for name, slug in OPENROUTER_BOTS.items():
    run_bot(name, clients.make_openrouter(slug, temperature=0.0, max_tokens=1000), SAMPLE)

gemini:   0%|          | 0/30 [00:00<?, ?it/s]

chatgpt:   0%|          | 0/30 [00:00<?, ?it/s]

claude:   0%|          | 0/30 [00:00<?, ?it/s]

deepseek:   0%|          | 0/30 [00:00<?, ?it/s]

In [8]:
# local B1 bot (float32, CPU)
local_fn = clients.make_local_hf(LOCAL_BOT[1], max_new_tokens=300, temperature=0.0)
run_bot(LOCAL_BOT[0], local_fn, SAMPLE if LOCAL_ON_EXTRA else [v for v in SAMPLE if v["set"] == "core"])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

local-qwen2.5-1.5b:   0%|          | 0/5 [00:00<?, ?it/s]

## Optional: prompt-engineering check
If a model refuses or won't give a ranked list, change `SYSTEM_DOCTOR` in `src/protocol.py` (or pass `system=`), re-run **only that bot** under a new bot name (e.g. `claude-v2`) and report the before/after.

In [9]:
runs = pd.DataFrame(store.load_runs())
display(runs.groupby(["bot", "set"]).size().unstack(fill_value=0))
core = runs[runs["set"] == "core"]
for _, r in core.sort_values(["vignette_id", "bot"]).iterrows():
    print(f"[{r['bot']} | vignette {r['vignette_id']}]\n{r['final_text']}\n" + "-"*70)

set,core,extra
bot,,
chatgpt,5,25
claude,5,25
deepseek,5,25
gemini,5,25
llama2-13b,3,0
llama2-13b-ollama,5,0
llama2-7b,5,0
llama2-7b-ollama,5,0
local-qwen2.5-1.5b,5,0


[chatgpt | vignette 15]
1. **Social Anxiety Disorder** - Characterized by intense fear of social situations and performance, leading to avoidance behaviors, which aligns with your anxiety in crowded places and public speaking.

2. **Panic Disorder** - Involves recurrent panic attacks and persistent worry about future attacks, which could explain your extreme anxiety and panic episodes.

3. **Generalized Anxiety Disorder** - Presents with excessive worry about various aspects of life, but the specific focus on social situations makes this less likely than the first two.

Most likely diagnosis: Social Anxiety Disorder.
----------------------------------------------------------------------
[claude | vignette 15]
Based on your presentation of extreme anxiety, panic attacks, and specific triggers (crowded places and public speaking), here are my top 3 differential diagnoses:

1. **Panic Disorder with Agoraphobia** - Recurrent panic attacks with avoidance of situations where escape might be 